1. Dendritic Tree GNN

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import MessagePassing
from torch_geometric.loader import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
import random
import networkx as nx
import matplotlib.pyplot as plt


def create_tree_graph(num_layers=3, max_branches=4):
    """Creates a tree-like graph with consistent node and edge indexing."""
    edge_index = []
    current_node = 0
    next_node = 1

    for layer in range(num_layers):
        layer_nodes = []
        for _ in range(max_branches**layer):
            num_children = random.randint(1, max_branches)
            for _ in range(num_children):
                edge_index.append([current_node, next_node])
                layer_nodes.append(next_node)
                next_node += 1
            current_node += 1

    # Determine the total number of nodes
    num_nodes = max(max(edge[0], edge[1]) for edge in edge_index) + 1

    # Create node feature matrix `x` with the correct number of rows
    x = torch.tensor([[random.randint(1, 50),  # n_e
                       random.randint(1, 50),  # n_i
                       random.uniform(0.5, 2.0),  # w_e
                       random.uniform(0.5, 2.0)]  # w_i
                      for _ in range(num_nodes)], dtype=torch.float)

    # Convert edge list to tensor
    edge_index = torch.tensor(edge_index, dtype=torch.long).t()

    # Validate that edge_index matches node range
    assert edge_index.max().item() < x.size(0), (
        f"Edge indices exceed the number of nodes in the feature matrix x. "
        f"Max edge index: {edge_index.max().item()}, Num nodes: {x.size(0)}"
    )

    # Generate random edge weights
    edge_attr = torch.rand(edge_index.size(1))

    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)

In [2]:
class SoftAdjacencyTreeGNN(MessagePassing):
    def __init__(self, num_nodes, num_layers):
        super().__init__(aggr='add')  # Aggregate activations via summation
        self.adjacency = torch.nn.Parameter(torch.rand(num_nodes, num_nodes))  # Soft adjacency matrix
        self.num_layers = num_layers
        self.node_layers = torch.arange(num_layers).repeat_interleave(num_nodes // num_layers)

    def enforce_tree_structure(self):
        """Enforce tree-like structure."""
        adjacency_mask = torch.zeros_like(self.adjacency)
        for l in range(self.num_layers - 1):
            layer_nodes = (self.node_layers == l).nonzero().flatten()
            next_layer_nodes = (self.node_layers == l + 1).nonzero().flatten()
            for i in layer_nodes:
                for j in next_layer_nodes:
                    adjacency_mask[i, j] = 1
        return self.adjacency * adjacency_mask

    def forward(self, x):
        adjacency = self.enforce_tree_structure()
        soft_adjacency = torch.sigmoid(adjacency)
        edge_index = torch.nonzero(soft_adjacency).t()
        edge_weight = soft_adjacency[edge_index[0], edge_index[1]]
        return self.propagate(edge_index, x=x, edge_weight=edge_weight)

    def message(self, x_j, edge_weight):
        return edge_weight.view(-1, 1) * x_j

    def update(self, aggr_out, x):
        n_e, n_i, w_e, w_i = x[:, 0], x[:, 1], x[:, 2], x[:, 3]
        aggr_out = F.relu(aggr_out)
        dendritic_activation = (n_e * w_e + aggr_out) / (n_e * w_e + aggr_out + n_i * w_i + 1)
        return dendritic_activation

In [3]:
class MNISTDendriticClassifier(nn.Module):
    def __init__(self, num_neurons, num_layers, max_branches):
        super().__init__()
        self.num_neurons = num_neurons
        self.dendritic_trees = nn.ModuleList(
            [SoftAdjacencyTreeGNN(num_nodes=100, num_layers=num_layers) for _ in range(num_neurons)]
        )
        self.fc = nn.Linear(num_neurons, 10)  # Fully connected layer for classification

    def forward(self, x):
        batch_size = x.size(0)
        soma_outputs = torch.zeros(batch_size, self.num_neurons, device=x.device)

        for i in range(self.num_neurons):
            dendritic_output = self.dendritic_trees[i](x)
            soma_outputs[:, i] = dendritic_output[0]  # Soma output

        return self.fc(soma_outputs)

In [4]:
class SoftAdjacencyTreeGNN(MessagePassing):
    def __init__(self, num_nodes, num_layers):
        super().__init__(aggr='add')  # Aggregate activations via summation
        self.adjacency = torch.nn.Parameter(torch.rand(num_nodes, num_nodes))  # Soft adjacency matrix
        self.num_layers = num_layers
        self.node_layers = torch.arange(num_layers).repeat_interleave(num_nodes // num_layers)

    def enforce_tree_structure(self):
        """Enforce tree-like structure."""
        adjacency_mask = torch.zeros_like(self.adjacency)
        for l in range(self.num_layers - 1):
            layer_nodes = (self.node_layers == l).nonzero().flatten()
            next_layer_nodes = (self.node_layers == l + 1).nonzero().flatten()
            for i in layer_nodes:
                for j in next_layer_nodes:
                    adjacency_mask[i, j] = 1
        return self.adjacency * adjacency_mask

    def forward(self, x):
        adjacency = self.enforce_tree_structure()
        soft_adjacency = torch.sigmoid(adjacency)

        # Generate edge_index from soft adjacency
        edge_index = torch.nonzero(soft_adjacency).t()
        
        # Explicitly clip edge_index to valid range
        edge_index = edge_index[:, edge_index[1] < x.size(0)]
        
        edge_weight = soft_adjacency[edge_index[0], edge_index[1]]
        return self.propagate(edge_index, x=x, edge_weight=edge_weight)
    
    def message(self, x_j, edge_weight):
        return edge_weight.view(-1, 1) * x_j

    def update(self, aggr_out, x):
        n_e, n_i, w_e, w_i = x[:, 0], x[:, 1], x[:, 2], x[:, 3]
        aggr_out = F.relu(aggr_out)
        dendritic_activation = (n_e * w_e + aggr_out) / (n_e * w_e + aggr_out + n_i * w_i + 1)
        return dendritic_activation

In [5]:
def create_genome(num_layers=3, max_branches=4):
    return {
        "structure": [[random.randint(1, max_branches) for _ in range(max_branches**layer)] for layer in range(num_layers)]
    }

In [6]:
def evaluate_fitness(genome, model, train_loader, test_loader):
    """Evaluate fitness of a genome based on model accuracy."""
    # Update the model structure
    for tree in model.trees:
        tree.edge_index = create_tree_from_genome(genome["structure"])
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
    train_model(model, train_loader, optimizer, epochs=1)
    accuracy = evaluate_model(model, test_loader)["accuracy"]
    return accuracy

In [7]:
def genetic_algorithm(model, train_loader, test_loader, population_size=10, generations=5):
    population = [create_genome() for _ in range(population_size)]
    for generation in range(generations):
        fitness_scores = [evaluate_fitness(genome, model, train_loader, test_loader) for genome in population]
        top_genomes = sorted(zip(population, fitness_scores), key=lambda x: x[1], reverse=True)[:population_size // 2]
        selected_population = [genome for genome, _ in top_genomes]
        offspring = []
        for _ in range(population_size - len(selected_population)):
            parent1, parent2 = random.sample(selected_population, 2)
            child_structure = crossover(parent1["structure"], parent2["structure"])
            offspring.append({"structure": child_structure})
        population = selected_population + offspring
    return max(population, key=lambda g: evaluate_fitness(g, model, train_loader, test_loader))

In [8]:
def adjacency_regularizer(adjacency, lambda_reg=0.1):
    """Regularize adjacency matrix to penalize unnecessary edges."""
    sparsity_penalty = lambda_reg * torch.sum(torch.abs(adjacency))
    return sparsity_penalty

In [9]:
def load_mnist(batch_size=64):
    train_dataset = MNIST(root='.', train=True, transform=ToTensor(), download=True)
    test_dataset = MNIST(root='.', train=False, transform=ToTensor(), download=True)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader, test_loader

In [10]:
def train_model(model, train_loader, optimizer, epochs=10, lambda_reg=0.1):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in train_loader:
            images = images.view(images.size(0), -1)  # Flatten images
            labels = labels.to(images.device)
            optimizer.zero_grad()
            outputs = model(images)  # Forward pass
            loss = F.cross_entropy(outputs, labels)  # Classification loss

            # Add adjacency regularization
            for tree in model.dendritic_trees:
                loss += adjacency_regularizer(tree.adjacency, lambda_reg)

            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss:.4f}")

In [11]:
def evaluate_model(model, test_loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.view(images.size(0), -1)
            labels = labels.to(images.device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

In [12]:
if __name__ == "__main__":
    
    data = create_tree_graph(num_layers=3, max_branches=4)
    print(f"Number of nodes in x: {data.x.size(0)}")
    print(f"Max edge index: {data.edge_index.max().item()}")
    print(f"Edge indices:\n{data.edge_index}")
    
    # Toggle between soft adjacency and genetic algorithm
    use_soft_adjacency = True  

    # Configuration
    num_neurons = 10
    num_layers = 3
    max_branches = 4
    epochs = 10
    batch_size = 64
    learning_rate = 0.01
    lambda_reg = 0.1

    # Load MNIST data
    train_loader, test_loader = load_mnist(batch_size)

    if use_soft_adjacency:
        print("\n=== Training with Soft Adjacency ===")
        # Initialize Soft Adjacency Model
        model = MNISTDendriticClassifier(num_neurons, num_layers, max_branches)
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

        # Train model
        for epoch in range(epochs):
            train_model(model, train_loader, optimizer, epochs=1, lambda_reg=lambda_reg)

        # Evaluate model
        accuracy = evaluate_model(model, test_loader)
        print(f"\nSoft Adjacency Model Test Accuracy: {accuracy:.2f}%")

    else:
        print("\n=== Training with Genetic Algorithm ===")
        # Initialize Genetic Algorithm Model
        model = MNISTDendriticClassifier(num_neurons, num_layers, max_branches)

        # Optimize using Genetic Algorithm
        best_genome = genetic_algorithm(model, train_loader, test_loader, population_size=10, generations=5)
        print(f"\nBest Genome: {best_genome}")

        # Fine-tune the model with the best genome
        for tree in model.trees:
            tree.edge_index = create_tree_from_genome(best_genome['structure'])

        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
        train_model(model, train_loader, optimizer, epochs=epochs)

        # Evaluate model
        accuracy = evaluate_model(model, test_loader)
        print(f"\nGenetic Algorithm Model Test Accuracy: {accuracy:.2f}%")

Number of nodes in x: 45
Max edge index: 44
Edge indices:
tensor([[ 0,  0,  1,  1,  1,  1,  2,  3,  4,  4,  4,  5,  5,  5,  6,  6,  6,  6,
          7,  7,  7,  8,  8,  8,  9, 10, 10, 10, 11, 12, 12, 13, 13, 14, 15, 16,
         17, 17, 18, 18, 18, 18, 19, 20],
        [ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
         19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
         37, 38, 39, 40, 41, 42, 43, 44]])

=== Training with Soft Adjacency ===


IndexError: Found indices in 'edge_index' that are larger than 63 (got 99). Please ensure that all indices in 'edge_index' point to valid indices in the interval [0, 64) in your node feature matrix and try again.